In [22]:
import sys

sys.path.append("../src")

import altair as alt

alt.renderers.enable("jupyter", offline=True)
alt.data_transformers.disable_max_rows()

from pathlib import Path

import altair as alt
import numpy as np
import polars as pl
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

from config import cfg
from data.data_process import add_fold
from data.simple_feature_eng import preprocess

# warnings.filterwarnings("ignore")
# warnings.simplefilter("ignore")


def plot_point(df: pl.DataFrame):
    transforms = [pl.col(col).cast(pl.Float64) for col in ["Episode_Length_minutes", "Listening_Time_minutes"]]
    df = df.with_columns(transforms)

    print("Length:", len(df))
    df.plot.point(
        x="Episode_Length_minutes",
        y="Listening_Time_minutes",
    ).properties(
        width=800,
        height=400,
    ).show()


def plot_linear_regression(df):
    X = df.select("Episode_Length_minutes").to_numpy()
    y = df.select("Listening_Time_minutes").to_numpy()

    model = LinearRegression()
    model.fit(X, y)
    predictions = model.predict(X)

    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y, predictions))

    print(f"Coefficient: {model.coef_[0][0]:.4f}")
    print(f"Intercept: {model.intercept_[0]:.4f}")
    print(f"RMSE: {rmse:.4f}")

    df = df.with_columns(pl.Series("Predicted", predictions.flatten()))
    if len(df) > 10000:
        df = df.sample(10000)

    scatter_plot = (
        alt.Chart(df.to_pandas())
        .mark_circle()
        .encode(
            x=alt.X("Episode_Length_minutes", title="Episode Length (minutes)"),
            y=alt.Y("Listening_Time_minutes", title="Listening Time (minutes)"),
            tooltip=["Episode_Length_minutes", "Listening_Time_minutes"],
        )
    )

    regression_line = alt.Chart(df.to_pandas()).mark_line(color="red").encode(x="Episode_Length_minutes", y="Predicted")

    chart = (scatter_plot + regression_line).properties(
        width=800,
        height=400,
    )

    return chart.show()


cfg.train_path = Path("../data/train.csv")
cfg.test_path = Path("../data/test.csv")
cfg.pltpd_path = Path("../data/podcast_dataset.csv")


df_test = pl.read_csv(cfg.test_path)

df_train = pl.read_csv(cfg.train_path)
df_train = df_train.drop_nulls(subset=["Episode_Length_minutes", "Number_of_Ads"])


# df_train = df_train.drop("id")
df_train = add_fold(df_train)
df_train = preprocess(df_train)

df_pltpd = pl.read_csv(cfg.pltpd_path)
df_pltpd = df_pltpd.drop_nulls(subset=["Episode_Length_minutes", "Number_of_Ads", "Listening_Time_minutes"])
df_pltpd = df_pltpd.with_columns(pl.col("Number_of_Ads").cast(pl.Float64))
df_pltpd = add_fold(df_pltpd)
df_pltpd = preprocess(df_pltpd)
df_pltpd = df_pltpd.with_columns(pl.Series(range(1_000_000, 1_000_000 + len(df_pltpd))).alias("id"))
# df_pltpd

df_train = df_train.drop(["id", "fold"])
df_pltpd = df_pltpd.drop(["id", "fold"])

df = df_train.clone()
df


Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Episode_Length_minutes_NaN,Guest_Popularity_percentage_NaN,Episode_Num_Cat
str,f64,str,f64,str,str,f64,f64,str,f64,i32,cat,cat,cat
"""1""",119.8,"""1""",66.95,"""5""","""14""",75.95,2.0,"""0""",88.01241,26,"""false""","""false""","""26"""
"""2""",73.9,"""2""",69.97,"""1""","""17""",8.97,0.0,"""0""",44.92531,16,"""false""","""false""","""16"""
"""3""",67.17,"""3""",57.22,"""0""","""10""",78.7,2.0,"""2""",46.27824,45,"""false""","""false""","""45"""
"""4""",110.51,"""4""",80.07,"""0""","""14""",58.68,3.0,"""1""",75.61031,86,"""false""","""false""","""86"""
"""5""",26.54,"""4""",48.96,"""5""","""14""",53.63,3.0,"""2""",22.77047,19,"""false""","""true""","""19"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""36""",75.66,"""2""",69.36,"""5""","""10""",53.63,0.0,"""0""",56.87058,25,"""false""","""true""","""25"""
"""19""",75.75,"""8""",35.21,"""5""","""21""",53.63,2.0,"""1""",45.46242,21,"""false""","""true""","""21"""
"""37""",30.98,"""9""",78.58,"""3""","""10""",84.89,0.0,"""0""",15.26,51,"""false""","""false""","""51"""


In [ ]:
df_train = df_train.with_columns(
    pl.col("Genre").cast(pl.Int32),
    pl.col("Publication_Day").cast(pl.Int32),
    pl.col("Publication_Time").cast(pl.Int32),
)

df_train.select(
    [
        "Episode_Length_minutes",
        "Listening_Time_minutes",
        "Genre",
        "Host_Popularity_percentage",
        "Guest_Popularity_percentage",
        "Publication_Day",
        "Publication_Time",
        "Number_of_Ads",
        "Episode_Sentiment",
        "Episode_Num",
    ]
)

In [13]:
plot_linear_regression(df_train)

Coefficient: 0.7583
Intercept: -3.1760
RMSE: 10.8908


JupyterChart(spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'layer': [{'mark': {…

In [14]:
df = df_train.filter(pl.col("Episode_Length_minutes") != pl.col("Listening_Time_minutes"))

plot_linear_regression(df)

Coefficient: 0.7587
Intercept: -3.2215
RMSE: 10.8887


JupyterChart(spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'layer': [{'mark': {…

In [ ]:
plot_linear_regression(df_pltpd)

Coefficient: 0.7388
Intercept: -0.8834
RMSE: 11.7217


JupyterChart(spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'layer': [{'mark': {…

In [16]:
df = df_pltpd.filter(pl.col("Episode_Length_minutes") != pl.col("Listening_Time_minutes"))

plot_linear_regression(df)

Coefficient: 0.7401
Intercept: -2.4876
RMSE: 11.2211


JupyterChart(spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'layer': [{'mark': {…

In [3]:
df["Listening_Time_minutes"].describe()

statistic,value
str,f64
"""count""",749999.0
"""null_count""",0.0
"""mean""",45.437435
"""std""",27.138313
"""min""",0.0
"""25%""",23.17835
"""50%""",43.37946
"""75%""",64.81158
"""max""",119.97


In [ ]:
import polars as pl
import numpy as np
from sklearn.manifold import TSNE
import altair as alt


features = df_train.to_numpy()

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_results = tsne.fit_transform(features)

df = df.with_columns([
    pl.Series("tsne_1", tsne_results[:, 0]),
    pl.Series("tsne_2", tsne_results[:, 1])
])

chart = alt.Chart(df).mark_circle(size=60).encode(
    x='tsne_1:Q',
    y='tsne_2:Q',
    color=alt.Color('label:N', scale=alt.Scale(scheme='category10')),
    tooltip=['label:N']
).properties(
    width=600,
    height=400,
    title='t-SNE Visualization of High-Dimensional Data'
).interactive()

chart.show()


Polarsデータフレームの形状: (662906, 14)
データフレームの先頭5行:
shape: (5, 14)
┌────────────┬────────────┬───────┬────────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ Podcast_Na ┆ Episode_Le ┆ Genre ┆ Host_Popul ┆ … ┆ Episode_N ┆ Episode_L ┆ Guest_Pop ┆ Episode_N │
│ me         ┆ ngth_minut ┆ ---   ┆ arity_perc ┆   ┆ um        ┆ ength_min ┆ ularity_p ┆ um_Cat    │
│ ---        ┆ es         ┆ str   ┆ entage     ┆   ┆ ---       ┆ utes_NaN  ┆ ercentage ┆ ---       │
│ str        ┆ ---        ┆       ┆ ---        ┆   ┆ i32       ┆ ---       ┆ _Na…      ┆ cat       │
│            ┆ f64        ┆       ┆ f64        ┆   ┆           ┆ cat       ┆ ---       ┆           │
│            ┆            ┆       ┆            ┆   ┆           ┆           ┆ cat       ┆           │
╞════════════╪════════════╪═══════╪════════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 1          ┆ 119.8      ┆ 1     ┆ 66.95      ┆ … ┆ 26        ┆ false     ┆ false     ┆ 26        │
│ 2          ┆ 73.9       ┆ 2  

ValueError: could not convert string to float: 'Joke Junction'

In [52]:
import numpy as np

def calculate_rmse(actual, predicted):
    squared_diff = (actual - predicted) ** 2
    mean_squared_diff = squared_diff.mean()
    rmse = np.sqrt(mean_squared_diff)
    return rmse

def optimize_scaling_factor(input_data, target_data, n_searches=20):
    initial_x = target_data.mean() / input_data.mean()
    
    # Define search range
    lower_bound = initial_x * 0.5
    upper_bound = initial_x * 1.5
    
    # Perform n binary searches
    for _ in range(n_searches):
        mid_point = (lower_bound + upper_bound) / 2
        
        delta = (upper_bound - lower_bound) * 0.1
        
        lower_x = mid_point - delta
        upper_x = mid_point + delta
        
        lower_rmse = calculate_rmse(target_data, input_data * lower_x)
        upper_rmse = calculate_rmse(target_data, input_data * upper_x)
        
        if lower_rmse < upper_rmse:
            upper_bound = mid_point
        else:
            lower_bound = mid_point
    
    best_x = (lower_bound + upper_bound) / 2
    best_rmse = calculate_rmse(target_data, input_data * best_x)
    
    return best_x, best_rmse

import time
for i in range(0, 100, 1):
    start_time = time.time()
    x_optimal = optimize_scaling_factor(df_ex["Episode_Length_minutes"], df_ex["Listening_Time_minutes"], n_searches=i)
    print(i, "\t", x_optimal, "\t",  time.time() - start_time)

0 	 (0.9598565775851021, 6.2916710331445485) 	 0.002655029296875
1 	 (0.7198924331888266, 14.23471322374616) 	 0.0017430782318115234
2 	 (0.8398745053869643, 8.918088931623851) 	 0.002685070037841797
3 	 (0.8998655414860333, 7.011795185255232) 	 0.0018329620361328125
4 	 (0.9298610595355676, 6.463500341295242) 	 0.0008518695831298828
5 	 (0.9448588185603348, 6.327036445085333) 	 0.0009577274322509766
6 	 (0.9523576980727184, 6.29649582675391) 	 0.0010669231414794922
7 	 (0.9561071378289103, 6.290857862490507) 	 0.0017609596252441406
8 	 (0.9579818577070063, 6.290457747135269) 	 0.003034830093383789
9 	 (0.9570444977679583, 6.290456119788039) 	 0.0019381046295166016
10 	 (0.9575131777374823, 6.290406510406307) 	 0.0019037723541259766
11 	 (0.9572788377527204, 6.290418709368783) 	 0.002056121826171875
12 	 (0.9573960077451014, 6.290409458439173) 	 0.0029158592224121094
13 	 (0.9574545927412919, 6.290407196559651) 	 0.002012968063354492
14 	 (0.9574838852393871, 6.2904066565171455) 	 0.00